In [ ]:
# Example 1: basic aggregate with default 'sum' function.
import numpy as np
import numpy_groupies as npg

group_idx = np.array([0, 0, 1, 1, 2, 2, 2])
values = np.array([1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0])

result = npg.aggregate(group_idx, values)
print(f"group_idx = {group_idx}")
print(f"values    = {values}")
print(f"sum per group = {result}")

# group 0: 1+2=3, group 1: 3+4=7, group 2: 5+6+7=18
np.testing.assert_array_equal(result, np.array([3.0, 7.0, 18.0]))
print("OK: default aggregate (sum) matches expected")


In [ ]:
# Example 2: aggregate with named functions (mean, max, min, prod).
import numpy as np
import numpy_groupies as npg

group_idx = np.array([0, 0, 1, 1, 2, 2])
values = np.array([1.0, 3.0, 2.0, 8.0, 4.0, 4.0])

means = npg.aggregate(group_idx, values, func='mean')
maxes = npg.aggregate(group_idx, values, func='max')
mins = npg.aggregate(group_idx, values, func='min')
prods = npg.aggregate(group_idx, values, func='prod')

print(f"mean per group = {means}")
print(f"max per group  = {maxes}")
print(f"min per group  = {mins}")
print(f"prod per group = {prods}")

np.testing.assert_allclose(means, [2.0, 5.0, 4.0])
np.testing.assert_array_equal(maxes, [3.0, 8.0, 4.0])
np.testing.assert_array_equal(mins, [1.0, 2.0, 4.0])
np.testing.assert_array_equal(prods, [3.0, 16.0, 16.0])
print("OK: named-function aggregates all correct")


In [ ]:
# Example 3: fill_value, dtype, and explicit size.
import numpy as np
import numpy_groupies as npg

# Note: no items in group 1; we ask for 4 output groups (0..3).
group_idx = np.array([0, 0, 2, 3])
values = np.array([10, 20, 30, 40])

result = npg.aggregate(group_idx, values, func='sum', size=4, fill_value=-1, dtype=np.int64)
print(f"result (size=4, fill=-1) = {result}")
print(f"dtype = {result.dtype}")

assert result.shape == (4,)
assert result.dtype == np.int64
assert result[1] == -1, f"missing group should be fill_value -1, got {result[1]}"
np.testing.assert_array_equal(result, np.array([30, -1, 30, 40], dtype=np.int64))
print("OK: size/fill_value/dtype honored")


In [ ]:
# Example 4: custom callable as aggregation function.
import numpy as np
import numpy_groupies as npg

group_idx = np.array([0, 0, 0, 1, 1, 2])
values = np.array([1.0, 2.0, 3.0, 10.0, 20.0, 100.0])

# range = max - min within each group
result = npg.aggregate(group_idx, values, func=lambda g: g.max() - g.min())
print(f"range per group = {result}")

# group 0: 3-1=2, group 1: 20-10=10, group 2: 100-100=0
np.testing.assert_allclose(result, [2.0, 10.0, 0.0])
print("OK: callable func produces per-group range")


In [ ]:
# Example 5: multi-dimensional group_idx (a tuple of index arrays produces an N-D output).
import numpy as np
import numpy_groupies as npg

# 2-D grouping: row_idx and col_idx pick cells in a 2x3 output grid.
row_idx = np.array([0, 0, 1, 1, 1])
col_idx = np.array([0, 1, 1, 2, 2])
values = np.array([5, 7, 3, 4, 6])

grid = npg.aggregate((row_idx, col_idx), values, func='sum', size=(2, 3), fill_value=0)
print(f"grid =\n{grid}")

expected = np.array([
    [5, 7, 0],
    [0, 3, 10],
])
np.testing.assert_array_equal(grid, expected)
assert grid.shape == (2, 3)
print("OK: 2-D aggregate matches expected grid")
